In [4]:
import os
import sys
import pandas as pd

# Отиваме една папка нагоре, за да намерим 'src'
project_root = os.path.dirname(os.path.abspath(""))
if project_root not in sys.path:
    sys.path.append(project_root)

# Импортираме само сплитера и пайплайна (без лоудъра)
from src.data.splitters import split_skab_groups
from src.data.preprocess import PreprocessingPipeline

print("📂 Зареждане на вече готовия файл от processed data...")

# 1. Подаваме директния път до вече съществуващия combined.csv
PROCESSED_FILE_PATH = "../data/processed/skab/combined.csv"

try:
    # Директно четем готовия записан файл от диска
    combined_df = pd.read_csv(PROCESSED_FILE_PATH)
    print(f"📦 Успешно прочетен combined.csv! Общ брой редове: {len(combined_df)}")
    
    print("-" * 60)
    
    # 2. Разрязваме готовите данни на 5 фолда с твоя StratifiedGroupKFold
    folds = split_skab_groups(combined_df)
    train_fold_1, test_fold_1 = folds[0] # Взимаме само ПЪРВИЯ фолд за теста
    
    # Отделяме валидационен сет от тренировъчния за нуждите на Early Stopping
    val_fold_1 = train_fold_1.sample(frac=0.2, random_state=42)
    train_fold_1 = train_fold_1.drop(val_fold_1.index)
    
    print("-" * 60)
    
    # 3. Извикваме твоя PreprocessingPipeline (Скалер + PCA)
    pipeline = PreprocessingPipeline(scaler_type="standard", pca_components=1)
    
    print("⚙️ Стартиране на fit_transform върху Fold 1...")
    processed_artifacts = pipeline.fit_transform(
        train_df=train_fold_1,
        validation_df=val_fold_1,
        test_df=test_fold_1
    )
    
    # 4. ДЕМОНСТРАЦИЯ НА КРАЙНИЯ РЕЗУЛТАТ
    print("\n📈 ТАБЛИЦА СЛЕД PCA (Вход за моделите - първите 5 реда):")
    display(processed_artifacts.train.head(5))

except FileNotFoundError:
    print(f"❌ Грешка: Не можах да намеря готовия файл на път: {PROCESSED_FILE_PATH}")
except Exception as e:
    print(f"❌ Възникна неочаквана софтуерна грешка: {e}")

📂 Зареждане на вече готовия файл от processed data...
📦 Успешно прочетен combined.csv! Общ брой редове: 22472
------------------------------------------------------------
✂️ Splitter: Initializing StratifiedGroupKFold with 5 splits...
   Fold 1: Train rows = 17974 | Test rows = 4498
   Fold 2: Train rows = 17958 | Test rows = 4514
   Fold 3: Train rows = 17975 | Test rows = 4497
   Fold 4: Train rows = 18037 | Test rows = 4435
   Fold 5: Train rows = 17944 | Test rows = 4528
✅ Splitter: Group-based cross-validation splits are ready.
------------------------------------------------------------
⚙️ Стартиране на fit_transform върху Fold 1...
⚙️ Pipeline: Executing PCA dimensionality reduction...
✅ Pipeline: Scaling and PCA compression successfully finished.

📈 ТАБЛИЦА СЛЕД PCA (Вход за моделите - първите 5 реда):


,PC1,datetime,anomaly,changepoint,source_group,source_file
1,4.765306,2020-03-09 10:14:34,0.0,0.0,valve1,0.csv
2,4.964040,2020-03-09 10:14:35,0.0,0.0,valve1,0.csv
4,4.532374,2020-03-09 10:14:37,0.0,0.0,valve1,0.csv
6,4.766108,2020-03-09 10:14:39,0.0,0.0,valve1,0.csv
7,4.955164,2020-03-09 10:14:40,0.0,0.0,valve1,0.csv
